# Feature Engineering
Builds and selects user, tweet, and graph features for the bot detection model.
Split into three sections:

Section 1 — Core Account + Tweet Behavior Features
- account age
- follower/friend ratio
- profile completeness
- tweet count
- retweet ratio
- hashtag/mention/URL averages
- text length averages
- save base feature dataset

Section 2 — Graph/Network Features
- load follower/friend/edge files
- standardize edges
- compute in-degree/out-degree
- compute graph degree
- compute reciprocity
- optional PageRank/community features
- merge with Round 1 dataset
- save graph-enhanced dataset

Section 3 — AI/Text Authenticity Features
- analyze tweet text for AI-like or synthetic-writing patterns
- compute per-account text authenticity features
- merge with Round 2 dataset
- save final model-ready dataset

These answer the three questions:
- Section 1: What does the account look like?
- Section 2: How is the account connected?
- Section 3: How natural or synthetic does the account’s language look?

In [2]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..")
sys.path.append(str(PROJECT_ROOT))

from utils.base_features import (
    ROUND1_FEATURE_COLS,
    available_feature_cols,
    build_round1_features_for_dataset,
    print_round1_summary,
    save_feature_frame,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

PREPROCESSED_DIR = PROJECT_ROOT / "data" / "pre-processed"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

ROUND1_DIR = PROCESSED_DIR / "round1_base"
ROUND1_DIR.mkdir(parents=True, exist_ok=True)

## Section 1: Base Features

In [ ]:
# Cell 2 — Dataset paths and reference dates

DATASETS = {
    "cresci_2015": {
        "users": PREPROCESSED_DIR / "cresci_2015" / "users_cresci_2015.csv",
        "tweets": PREPROCESSED_DIR / "cresci_2015" / "tweets_cresci_2015.csv",
        "reference_date": "2015-12-31",
    },
    "cresci_2017": {
        "users": PREPROCESSED_DIR / "cresci_2017" / "users_cresci_2017.csv",
        "tweets": PREPROCESSED_DIR / "cresci_2017" / "tweets_cresci_2017.csv",
        "reference_date": "2017-12-31",
    },
    "twibot_2020": {
        "users": PREPROCESSED_DIR / "twibot_2020" / "users_twibot_20.csv",
        "tweets": PREPROCESSED_DIR / "twibot_2020" / "tweets_twibot_20.csv",
        "reference_date": "2020-12-31",
    },
    "twibot_2022": {
        "users": PREPROCESSED_DIR / "twibot_2022" / "users_twibot_22.csv",
        "tweets": PREPROCESSED_DIR / "twibot_2022" / "tweets_twibot_22.csv",
        "reference_date": "2022-12-31",
    },
}

for name, paths in DATASETS.items():
    print(name)
    print("  users:", paths["users"].exists(), paths["users"])
    print("  tweets:", paths["tweets"].exists(), paths["tweets"])

In [ ]:
# Cell 3 — Build and save Round 1 features per dataset

round1_frames = []

for dataset_name, paths in DATASETS.items():
    print(f"\nBuilding Round 1 features for {dataset_name}")

    round1 = build_round1_features_for_dataset(dataset_name, paths)

    out_path = ROUND1_DIR / f"{dataset_name}_base_features.csv"
    save_feature_frame(round1, out_path)

    print(f"  Shape: {round1.shape}")
    print(f"  Saved: {out_path}")

    round1_frames.append(round1)

In [ ]:
# Cell 4 — Combine and save all Round 1 features

round1_all = pd.concat(round1_frames, ignore_index=True, sort=False)

combined_path = ROUND1_DIR / "all_datasets_base_features.csv"
save_feature_frame(round1_all, combined_path)

print(f"Combined Round 1 shape: {round1_all.shape}")
print(f"Saved combined Round 1 file to: {combined_path}")

In [ ]:
# Cell 5 — Round 1 sanity checks (clean ML-ready dataset)

print_round1_summary(round1_all)

round1_feature_cols = available_feature_cols(round1_all)

print("\nFinal feature columns:")
for col in sorted(round1_all.columns):
    print("-", col)

print("\nMissing values in feature columns:")
display(
    round1_all[round1_feature_cols]
    .isna()
    .sum()
    .to_frame("missing_count")
)

In [ ]:
# Cell 6 — Preview Round 1 feature behavior

summary_cols = [
    "account_age_days",
    "ff_ratio",
    "tweet_count_actual",
    "tweets_per_day",
    "avg_text_length",
    "retweet_ratio",
    "avg_hashtags",
    "avg_mentions",
    "avg_urls",
]

summary_cols = [c for c in summary_cols if c in round1_all.columns]

round1_summary = (
    round1_all
    .groupby(["dataset", "label"])[summary_cols]
    .mean()
    .round(3)
)

display(round1_summary)